# TTC Station Reliability — PipelineBuilds every table from `data/raw/` in order. Run top to bottom.`raw_*` tables are never modified. All cleaning happens in derived layers,so a bug means re-running one cell rather than re-downloading source files.| Layer | Tables ||---|---|| Raw | `routes`, `stops`, `stop_times`, `trips`, `raw_delays` || Denominator | `subway_stop_times`, `station_trips`, `station_trips_clean` || Clean | `clean_delays` || Analysis | `station_unreliability`, `station_unreliability_2018` || Output | `station_ci`, `station_data_final`, `rank_comparison` |

## 0. Setup

In [ ]:
import osimport globimport duckdbimport pandas as pdimport numpy as npos.makedirs("output", exist_ok=True)con = duckdb.connect("ttc.duckdb")

## 1. Load GTFSThe schedule feed provides the denominator: how many trains are scheduledto arrive at each station. Chosen over ridership data, which stops at 2019and cannot be projected across COVID.

In [ ]:
con.sql("CREATE OR REPLACE TABLE routes     AS SELECT * FROM read_csv_auto('data/raw/schedules/routes.txt')")con.sql("CREATE OR REPLACE TABLE stops      AS SELECT * FROM read_csv_auto('data/raw/schedules/stops.txt')")con.sql("CREATE OR REPLACE TABLE stop_times AS SELECT * FROM read_csv_auto('data/raw/schedules/stop_times.txt')")con.sql("CREATE OR REPLACE TABLE trips      AS SELECT * FROM read_csv_auto('data/raw/schedules/trips.txt')")# route_type = 1 is subway: routes 1, 2, 4. Line 3 closed 2023,# Lines 5/6 too recent for stable rates.con.sql("SELECT route_id, route_short_name, route_long_name FROM routes WHERE route_type = 1").df()

## 2. Denominator — scheduled trips per station

In [ ]:
# One row per scheduled train arrival at a subway platformcon.sql("""    CREATE OR REPLACE TABLE subway_stop_times AS    SELECT stop_times.* FROM trips    JOIN stop_times        ON trips.trip_id = stop_times.trip_id    WHERE trips.route_id IN [1,2,4]""")# GTFS has no parent_station for TTC, so stations are grouped on stop_name.# It splits Bloor-Yonge into "Bloor" (Line 2) and "Yonge" (Line 1) — merged here.mapping = pd.DataFrame([    ("Bloor Station", "Bloor-Yonge Station"),    ("Yonge Station", "Bloor-Yonge Station")], columns=["raw_name", "canonical_name"])con.sql("""    CREATE OR REPLACE TABLE station_trips AS        SELECT            COALESCE(mapping.canonical_name, SPLIT_PART(stop_name, ' -', 1)) AS stations,            COUNT(*) AS scheduled_trips        FROM stops        JOIN subway_stop_times            ON subway_stop_times.stop_id = stops.stop_id        LEFT JOIN mapping ON SPLIT_PART(stop_name, ' -', 1) = mapping.raw_name        GROUP BY 1""")# normalise to the delay data's vocabulary: uppercase, no ' Station' suffixcon.sql("""    CREATE OR REPLACE TABLE station_trips_clean AS        WITH        s1 AS (            SELECT * REPLACE (SPLIT_PART(stations, ' Station', 1) AS stations) FROM station_trips),        s2 AS (            SELECT * REPLACE (UPPER(stations) AS stations) FROM s1),        s3 AS (            SELECT * RENAME (stations AS station) FROM s2)    SELECT * FROM s3""")con.sql("SELECT COUNT(*) AS stations FROM station_trips_clean").df()

## 3. Load delay data10 files, 2014 to June 2026, mixed xlsx and csv. Column schema is stableacross all of them; the csv carries one extra `_id` column from CKAN.

In [ ]:
paths = [p for p in glob.glob("data/raw/delay/*") if os.path.isfile(p)]dfs = []for f in paths:    if os.path.splitext(f)[1].lower() == '.xlsx':        d = pd.read_excel(f)    else:        d = pd.read_csv(f)    dfs.append(d)all_delays = pd.concat(dfs, ignore_index=True)con.sql("CREATE OR REPLACE TABLE raw_delays AS SELECT * FROM all_delays")con.sql("SELECT COUNT(*) AS rows FROM raw_delays").df()

## 4. Clean layerThe delay data holds 1,853 distinct station values against a true count of 70.Causes: line suffixes (BD/YU/SRT), a 22-character truncation limit in the sourcesystem, station renames, and free-text entries for segments and facilities.Each CTE applies one rule. Order matters — see the comments.

In [ ]:
con.sql("""CREATE OR REPLACE TABLE clean_delays AS    WITH    -- Line 3 closed 2023 and is absent from GTFS, so it has no denominator.    -- Must drop before s4, or ' SRT' is stripped and Kennedy SRT merges into Kennedy.    s1 AS (        SELECT * FROM raw_delays WHERE Line IS DISTINCT FROM 'SRT'),    s2 AS (        SELECT * REPLACE (REGEXP_REPLACE(Station, '\\s+', ' ', 'g') AS Station) FROM s1),    -- splits at ' STATION', which also removes trailing descriptors    -- e.g. 'ROYAL YORK STATION (AP' -> 'ROYAL YORK'    s3 AS (        SELECT * REPLACE (SPLIT_PART(Station, ' STATION', 1) AS Station) FROM s2),    -- line suffixes are redundant; the Line column already carries this    s4 AS (        SELECT * REPLACE (REGEXP_REPLACE(Station, ' (BD|YU|YUS|SRT)$', '') AS Station) FROM s3),    s5 AS (        SELECT * REPLACE (REGEXP_REPLACE(Station, 'SHEPPARDSTATION$', 'SHEPPARD-YONGE') AS Station) FROM s4),    s6 AS (        SELECT * REPLACE (REGEXP_REPLACE(Station, 'BLOOR YONGE', 'BLOOR-YONGE') AS Station) FROM s5),    s7 AS (        SELECT * RENAME (Station AS station) FROM s6),    s8 AS (        SELECT * REPLACE (REPLACE(station, '.', '') AS station) FROM s7),    -- 2023 renames. DUNDAS is anchored: DUNDAS WEST is a different station on Line 2.    s9 AS (        SELECT * REPLACE (REPLACE(station, 'EGLINTON WEST', 'CEDARVALE') AS station) FROM s8),    s10 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^DUNDAS$', 'TMU') AS station) FROM s9),    -- TTC logs Bloor-Yonge under either half's name    s11 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^BLOOR$', 'BLOOR-YONGE') AS station) FROM s10),    s12 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^YONGE$', 'BLOOR-YONGE') AS station) FROM s11),    -- bare SHEPPARD assumed Sheppard-Yonge: Sheppard West is logged under its own name    s13 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^SHEPPARD$', 'SHEPPARD-YONGE') AS station) FROM s12),    s14 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^VAUGHAN MC$', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s13),    s15 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^NORTH YORK CTR$', 'NORTH YORK CENTRE') AS station) FROM s14),    s16 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^VMC$', 'VAUGHAN METROPOLITAN CENTRE') AS station) FROM s15),    s17 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, '^YONGE SH(P|EP)$', 'SHEPPARD-YONGE') AS station) FROM s16),    -- source truncates at 22 chars, cutting ' STATION' mid-word    s18 AS (        SELECT * REPLACE (REGEXP_REPLACE(station, ' STATIO?$', '') AS station) FROM s17),    -- xlsx files parse Date as a timestamp, the csv as text; split handles both    s19 AS (        SELECT * EXCLUDE(Date), SPLIT_PART(Date, ' ', 1) AS Date_normalized FROM s18),    s20 AS (        SELECT * EXCLUDE(_id, Time, Date_normalized),               STRPTIME(Date_normalized || ' ' || Time, '%Y-%m-%d %H:%M') AS Timestamp FROM s19),    -- Exclusion step. Drops yards, hostlers, carhouses, wyes, portals,    -- two-station segments and line-level records (~15k rows, 8%) —    -- none of which can be attributed to a single station.    s21 AS (        SELECT * EXCLUDE(t1.station, scheduled_trips) FROM s20        LEFT JOIN station_trips_clean t1 ON s20.station = t1.station        WHERE t1.station IS NOT NULL)SELECT * FROM s21""")con.sql("""    SELECT COUNT(*) AS rows,           COUNT(DISTINCT station) AS stations,           SUM(CASE WHEN Timestamp IS NULL THEN 1 ELSE 0 END) AS null_timestamps,           MIN(Timestamp) AS first, MAX(Timestamp) AS last    FROM clean_delays""").df()

## 5. Analysis`delay_rate` = delay incidents / scheduled trips. Zero-minute rows are excluded —they are logged incidents with no measurable service impact, and 98% of themalso have zero gap.Two versions: full range, and 2018 onward. The Vaughan extension openedDecember 2017, so its six stations have delays for only part of the full windowwhile carrying the same denominator. The 2018 version is the primary result.

In [ ]:
metrics = """    WITH    s1 AS (        SELECT station,            COUNT("Min Delay") AS delays,            SUM("Min Delay") AS total_delay,            AVG("Min Delay") AS avg_delay,            PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY "Min Delay") AS p50,            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "Min Delay") AS p95        FROM clean_delays        WHERE "Min Delay" > 0 {date_filter}        GROUP BY station),    s2 AS (        SELECT            t1.station,            delays / scheduled_trips AS delay_rate,            delays, total_delay, avg_delay, p50, p95        FROM s1 t1        JOIN station_trips_clean t2            ON t1.station = t2.station)    SELECT *,        RANK() OVER (ORDER BY delay_rate DESC) AS adjusted_rank,        RANK() OVER (ORDER BY delays DESC)     AS raw_rank,        raw_rank - adjusted_rank AS delta    FROM s2"""con.sql("CREATE OR REPLACE TABLE station_unreliability AS "        + metrics.format(date_filter=""))con.sql("CREATE OR REPLACE TABLE station_unreliability_2018 AS "        + metrics.format(date_filter="AND Timestamp >= '2018-01-01'"))con.sql("SELECT station, delay_rate, delays, raw_rank, adjusted_rank, delta "        "FROM station_unreliability_2018 ORDER BY adjusted_rank LIMIT 10").df()

## 6. Confidence intervalsParametric bootstrap. Delay counts are modelled as Poisson, so the observedcount is the only parameter needed — Poisson variance equals its mean.5,000 simulated counts per station, converted to rates, then the 2.5th and97.5th percentiles give a 95% interval.Caveat: Poisson assumes independent events. Delays cluster (one incidentgenerates several records), so counts are likely overdispersed and theseintervals are somewhat too narrow. A negative binomial model would widen them.

In [ ]:
rng = np.random.default_rng(1)bootstrap_results = con.sql("""    SELECT t1.station, delays, scheduled_trips    FROM station_unreliability_2018 t1    JOIN station_trips_clean t2 ON t1.station = t2.station""").df()sims  = rng.poisson(bootstrap_results['delays'].values,                    size=(5000, len(bootstrap_results)))rates = sims / bootstrap_results['scheduled_trips'].valuesbootstrap_results['ci_low'], bootstrap_results['ci_high'] = np.percentile(rates, [2.5, 97.5], axis=0)bootstrap_results['rate'] = bootstrap_results['delays'] / bootstrap_results['scheduled_trips']baseline = bootstrap_results['delays'].sum() / bootstrap_results['scheduled_trips'].sum()bootstrap_results['worse_than_baseline'] = bootstrap_results['ci_low'] > baselineprint(f"network baseline rate: {baseline:.4f}")print(f"significantly worse than baseline: {bootstrap_results['worse_than_baseline'].sum()} of {len(bootstrap_results)}")bootstrap_results.sort_values('rate', ascending=False).head(10)

## 7. Export

In [ ]:
con.sql("CREATE OR REPLACE TABLE station_ci AS SELECT * FROM bootstrap_results")con.sql("""    CREATE OR REPLACE TABLE station_data_final AS        SELECT * EXCLUDE(t2.station, t2.delays, rate)        FROM station_unreliability_2018 t1        JOIN station_ci t2 ON t1.station = t2.station""")# long format for the Tableau slope chart: one row per station per rank typecon.sql("""    CREATE OR REPLACE TABLE rank_comparison AS        SELECT station, raw_rank      AS rank_value, 'raw'      AS rank_type FROM station_data_final        UNION ALL        SELECT station, adjusted_rank AS rank_value, 'adjusted' AS rank_type FROM station_data_final""")con.sql("COPY station_data_final TO 'output/station_final.csv'    (HEADER, DELIMITER ',')")con.sql("COPY rank_comparison    TO 'output/rank_comparison.csv' (HEADER, DELIMITER ',')")print("exported to output/")